In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


url='https://saudi.tanqeeb.com/jobs-in-saudi/all/jobs/021225095.html?change_lang=1'

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")


# ============================================================
# 1. BASIC JOB INFORMATION
# ============================================================

# Job Title
title_element = soup.select_one("h3.job-title-text")
job_title = title_element.get_text(" ", strip=True) if title_element else None


# Work Mode + Job Type
tags = soup.select(".job-tags .job-tag")

work_mode = None
job_type = None

for tag in tags:
    value = tag.get_text(" ", strip=True)

    if value in ["On-site", "Remote", "Hybrid"]:
        work_mode = value

    elif value in ["Full Time", "Part Time", "Contract", "Internship"]:
        job_type = value


# ============================================================
# 2. LOCATION
# ============================================================

location_element = soup.select_one(".job-meta-item span")

country = None
city = None

if location_element:

    location_text = location_element.get_text(" ", strip=True)

    # Example:
    # "Saudi, Makkah"

    parts = [x.strip() for x in location_text.split(",")]

    if len(parts) >= 1:
        country = parts[0]

    if len(parts) >= 2:
        city = parts[1]


# ============================================================
# 3. POSTED DATE
# ============================================================

date_element = soup.select_one(".job-date")

posted_date = None

if date_element:
    posted_date = date_element.get("data-datetime")


# ============================================================
# 4. JOB ID
# ============================================================

apply_button = soup.select_one(".apply-btn")

job_id = None

if apply_button:
    job_id = apply_button.get("data-job-id")


# ============================================================
# 5. JOB URL
# ============================================================

# The actual page URL
job_url = url


# ============================================================
# 6. JOB DETAILS
# ============================================================

job_details = {}

meta_rows = soup.select(".meta-data .meta")

for row in meta_rows:

    spans = row.find_all("span")

    if len(spans) >= 2:

        label = spans[0].get_text(" ", strip=True)
        value = spans[1].get_text(" ", strip=True)

        job_details[label] = value


experience = job_details.get("Experience")
employment = job_details.get("Employment")
gender = job_details.get("Gender")
education = job_details.get("Education")
industry = job_details.get("Industry")


# ============================================================
# 7. JOB DESCRIPTION
# ============================================================

description_element = soup.select_one("#jobDescriptionBody")

job_description = (
    description_element.get_text(" ", strip=True)
    if description_element
    else None
)


# ============================================================
# 8. CREATE ONE JOB RECORD
# ============================================================

job = {
    "job_id": job_id,
    "job_title": job_title,
    "work_mode": work_mode,
    "job_type": job_type,
    "country": country,
    "city": city,
    "posted_date": posted_date,
    "experience": experience,
    "employment": employment,
    "gender": gender,
    "education": education,
    "industry": industry,
    "description": job_description,
    "job_url": job_url
}


# ============================================================
# 9. CREATE DATAFRAME
# ============================================================

df = pd.DataFrame([job])

print(df)


# ============================================================
# 10. SAVE TO CSV
# ============================================================

df.to_csv("tanqeeb_jobs.csv", index=False, encoding="utf-8-sig")

     job_id                                         job_title work_mode  \
0  21225095  Front Office Manager required for work in Makkah   On-site   

    job_type country    city                posted_date    experience  \
0  Full Time   Saudi  Makkah  2026-09-10T14:47:49+03:00  2 to 5 years   

  employment gender                   education  \
0  Full Time   Male  جامعي (بكالوريوس / ليسانس)   

                                  industry  \
0  Hospitality Services & Hotel Operations   

                                         description  \
0  A hotel management in Makkah announces a job v...   

                                             job_url  
0  https://saudi.tanqeeb.com/jobs-in-saudi/all/jo...  


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from urllib.parse import urljoin
import time


# ============================================================
# 1. MAIN SEARCH PAGE
# ============================================================

BASE_SEARCH_URL = (
    "https://saudi.tanqeeb.com/jobs/search?"
    "keywords=&country=54&state=0&category=-1&"
    "workplace=0&search_period=0&lang=all&change_lang=1"
)

BASE_URL = "https://saudi.tanqeeb.com"

# Safety cap in case pagination never signals "end" cleanly.
# Raise/lower as needed.
MAX_PAGES = 200


# ============================================================
# 2. OPEN SEARCH PAGE WITH SELENIUM AND WALK THROUGH PAGES
# ============================================================

def build_page_url(base_url, page_num):
    """
    Tanqeeb paginates with a `&page=N` query param.
    Page 1 is just the base URL (no param needed), but adding
    `&page=1` explicitly is harmless and keeps the loop uniform.
    """
    return f"{base_url}&page={page_num}"


def get_job_links_from_html(html):
    soup = BeautifulSoup(html, "html.parser")
    return soup.select("h2.search-job-title a.search-job-title-link")


options = Options()
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)

job_urls = []
page_num = 1

try:
    while page_num <= MAX_PAGES:

        page_url = build_page_url(BASE_SEARCH_URL, page_num)

        print(f"\nLoading search page {page_num}: {page_url}")

        driver.get(page_url)

        # Wait for page to load / JS to render results
        time.sleep(5)

        html = driver.page_source

        job_links = get_job_links_from_html(html)

        print(f"  -> Job links found on this page: {len(job_links)}")

        # Stop condition: no job cards on this page means we've
        # gone past the last page of results.
        if not job_links:
            print("No more job listings found. Stopping pagination.")
            break

        page_had_new_url = False

        for link in job_links:
            href = link.get("href")
            if href:
                full_url = urljoin(BASE_URL, href)
                if full_url not in job_urls:
                    job_urls.append(full_url)
                    page_had_new_url = True

        # Extra safety: if a page returns only links we've already
        # seen (e.g. the site clamped an out-of-range page back to
        # the last page), treat that as the end too.
        if not page_had_new_url:
            print("No new job URLs on this page (likely repeated last page). Stopping.")
            break

        page_num += 1

finally:
    driver.quit()


# Remove duplicates while preserving order (belt-and-suspenders,
# since we already dedupe while collecting above)
job_urls = list(dict.fromkeys(job_urls))

print("\nUnique job URLs collected across all pages:", len(job_urls))


# ============================================================
# 3. SCRAPE EACH JOB PAGE
# ============================================================

all_jobs = []

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/140.0.0.0 Safari/537.36"
    )
}

for i, url in enumerate(job_urls, start=1):

    print(f"\n[{i}/{len(job_urls)}] Scraping:")
    print(url)

    try:
        response = requests.get(url, headers=headers, timeout=30)

        if response.status_code != 200:
            print("Failed:", response.status_code)
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        # ====================================================
        # JOB ID
        # ====================================================
        apply_button = soup.select_one(".apply-btn")
        job_id = apply_button.get("data-job-id") if apply_button else None

        # ====================================================
        # JOB TITLE
        # ====================================================
        title_element = soup.select_one("h3.job-title-text")
        job_title = title_element.get_text(" ", strip=True) if title_element else None

        # ====================================================
        # WORK MODE + JOB TYPE
        # ====================================================
        tags = soup.select(".job-tags .job-tag")
        work_mode = None
        job_type = None

        for tag in tags:
            value = tag.get_text(" ", strip=True)
            value_lower = value.lower()

            if value_lower in ["on-site", "remote", "hybrid"]:
                work_mode = value
            elif value_lower in [
                "full time", "part time", "contract",
                "internship", "temporary", "freelance"
            ]:
                job_type = value

        # ====================================================
        # LOCATION
        # ====================================================
        location_element = soup.select_one(".job-meta-item span")
        country = None
        city = None

        if location_element:
            location_text = location_element.get_text(" ", strip=True)
            parts = [x.strip() for x in location_text.split(",")]

            if len(parts) >= 1:
                country = parts[0]
            if len(parts) >= 2:
                city = parts[1]

        # ====================================================
        # POSTED DATE
        # ====================================================
        date_element = soup.select_one(".job-date")
        posted_date = date_element.get("data-datetime") if date_element else None

        # ====================================================
        # JOB DETAILS
        # ====================================================
        job_details = {}
        meta_rows = soup.select(".meta-data .meta")

        for row in meta_rows:
            spans = row.find_all("span")
            if len(spans) >= 2:
                label = spans[0].get_text(" ", strip=True)
                value = spans[1].get_text(" ", strip=True)
                job_details[label] = value

        experience = job_details.get("Experience")
        employment = job_details.get("Employment")
        gender = job_details.get("Gender")
        education = job_details.get("Education")
        industry = job_details.get("Industry")

        # ====================================================
        # JOB DESCRIPTION
        # ====================================================
        description_element = soup.select_one("#jobDescriptionBody")
        description = (
            description_element.get_text(" ", strip=True)
            if description_element else None
        )

        # ====================================================
        # SAVE JOB
        # ====================================================
        job = {
            "job_id": job_id,
            "job_title": job_title,
            "work_mode": work_mode,
            "job_type": job_type,
            "country": country,
            "city": city,
            "posted_date": posted_date,
            "experience": experience,
            "employment": employment,
            "gender": gender,
            "education": education,
            "industry": industry,
            "description": description,
            "job_url": url
        }

        all_jobs.append(job)

        print("✓ Scraped:", job_title)

    except Exception as e:
        print("Error:", e)

    # Wait between requests
    time.sleep(1)


# ============================================================
# 4. CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(all_jobs)


# ============================================================
# 5. SAVE CSV
# ============================================================

df.to_csv("tanqeeb_jobs.csv", index=False, encoding="utf-8-sig")


# ============================================================
# 6. RESULTS
# ============================================================

print("\n" + "=" * 60)
print("Total URLs:", len(job_urls))
print("Successfully scraped:", len(df))
print("CSV saved as: tanqeeb_jobs.csv")
print("=" * 60)

df.head()

Job links found: 27
Unique job URLs: 27

[1/27] Scraping:
https://saudi.tanqeeb.com/jobs-in-saudi/all/jobs/021225095.html
✓ Scraped: Front Office Manager required for work in Makkah

[2/27] Scraping:
https://saudi.tanqeeb.com/jobs-in-saudi/all/jobs/021223553.html
✓ Scraped: Real Estate Sales Representative Wanted

[3/27] Scraping:
https://saudi.tanqeeb.com/jobs-in-saudi/all/jobs/021223456.html
✓ Scraped: Computer Programmer

[4/27] Scraping:
https://saudi.tanqeeb.com/jobs-in-saudi/all/jobs/021221858.html
✓ Scraped: Graphic Designer

[5/27] Scraping:
https://saudi.tanqeeb.com/jobs-in-saudi/all/jobs/021221113.html
✓ Scraped: Sales Manager Required – Hotel Supplies

[6/27] Scraping:
https://saudi.tanqeeb.com/jobs-in-saudi/all/jobs/021220909.html
✓ Scraped: Dental Nurse

[7/27] Scraping:
https://saudi.tanqeeb.com/jobs-in-saudi/all/jobs/021220908.html
✓ Scraped: Construction Site Supervisor

[8/27] Scraping:
https://saudi.tanqeeb.com/jobs-in-saudi/all/jobs/021226229.html
✓ Scraped: AFR Spec

,job_id,job_title,work_mode,job_type,country,city,posted_date,experience,employment,gender,education,industry,description,job_url
0,21225095,Front Office Manager required for work in Makkah,On-site,Full Time,Saudi,Makkah,2026-09-10T14:47:49+03:00,2 to 5 years,Full Time,Male,جامعي (بكالوريوس / ليسانس),Hospitality Services & Hotel Operations,A hotel management in Makkah announces a job v...,https://saudi.tanqeeb.com/jobs-in-saudi/all/jo...
1,21223553,Real Estate Sales Representative Wanted,On-site,Full Time,Saudi,Medina,2026-09-09T17:38:43+03:00,2 to 4 years,Full Time,Male,جامعي (بكالوريوس / ليسانس),Real Estate,Imtilak Global Real Estate is looking for new ...,https://saudi.tanqeeb.com/jobs-in-saudi/all/jo...
2,21223456,Computer Programmer,On-site,Full Time,Saudi,Riyadh,2026-09-09T14:24:19+03:00,1 to 6 years,Full Time,None,جامعي (بكالوريوس / ليسانس),None,"Designing, developing, and maintaining softwar...",https://saudi.tanqeeb.com/jobs-in-saudi/all/jo...
3,21221858,Graphic Designer,On-site,Full Time,Saudi,Riyadh,2026-09-08T16:56:35+03:00,1 to 5 years,Full Time,None,جامعي (بكالوريوس / ليسانس),None,At least one year of experience in the same fi...,https://saudi.tanqeeb.com/jobs-in-saudi/all/jo...
4,21221113,Sales Manager Required – Hotel Supplies,On-site,Full Time,Saudi,Riyadh,2026-09-08T14:05:12+03:00,5 to 10 years,Full Time,None,جامعي (بكالوريوس / ليسانس),Hospitality Services & Hotel Operations,A specialized marketing company operating in t...,https://saudi.tanqeeb.com/jobs-in-saudi/all/jo...
